# TEMA trend strategy with HMM regimes

A walk-forward hidden Markov model classifies regimes for a triple exponential moving average trend strategy. Includes fold diagnostics and backtest visualization.

Daily OHLCV is downloaded from Yahoo Finance when the notebook runs. Open this notebook from the `notebooks/` directory so it can import the shared data loader.


In [ ]:
"""
HMM STRATEGY: TEMA + YANG-ZHANG VOLATILITY PERCENTILES (SHORT-ENABLED VERSION)
==============================================================================

CRITICAL IMPROVEMENTS - VERSION 3.0:
====================================

**NEW IN VERSION 3.0: SHORT POSITION CAPABILITY**

1. **SHORT POSITIONS ENABLED**
   - Bearish regime: -50% SPY (short position)
   - Proper borrow cost modeling (1% annual)
   - Leverage limits enforced (max 100% gross)
   - Open-to-Open returns maintained for shorts
   
2. **SEMANTIC HMM INITIALIZATION**
   - HMM initialized with quantile-based means
   - Bearish: Low TEMA (25th percentile) + High vol (75th percentile)
   - Neutral: Median TEMA + Median vol
   - Bullish: High TEMA (75th percentile) + Low vol (25th percentile)
   - Encourages discovery of all 3 regimes
   
3. **FEATURE-BASED SHORT OVERLAY**
   - Backup mechanism if HMM doesn't detect Bearish regime
   - Forces short when TEMA < 0 AND volatility percentile >= 80%
   - Maintains T-1 to T signal flow (no look-ahead)
   - Configurable thresholds
   
4. **COMPREHENSIVE VALIDATION**
   - Regime vs Forward Returns matrix (1d, 3d, 5d, 10d, 21d)
   - Short position effectiveness analysis
   - Leverage limit compliance checks
   - Borrow cost tracking
   
5. **REALISTIC SHORT MECHANICS**
   - Daily borrow cost: |notional| * daily_borrow_rate
   - Only positive cash earns risk-free rate
   - Negative positions properly valued
   - All costs explicit and tracked

PREVIOUS IMPROVEMENTS (V2.0):
=============================

1. **ROLLING PERCENTILE WINDOW**
   - Changed from expanding to rolling 252-day window
   - Prevents early crisis periods from anchoring percentiles
   
2. **FEATURE-BASED REGIME ANCHORING**
   - Labels assigned based on feature statistics
   - Prevents "label switching" across folds
   
3. **OPEN-TO-OPEN RETURN ALIGNMENT**
   - Execution at Open[T] using signal from T-1
   - Portfolio valued at Open[T]
   - Returns: Open[T]/Open[T-1] - 1
   
4. **EVENT-DRIVEN REBALANCING**
   - Rebalances on regime change
   - Rebalances when allocation drifts >5%
   - Minimum 5 days between rebalances

WHY SHORT CAPABILITY MATTERS:
=============================

**Problem**: Strategy only uses 2 regimes (Bullish/Neutral) because 
the 2020-2024 period lacks sustained "downtrend + high volatility".
Without shorts, strategy is limited to long/cash.

**Solution**: Enable shorts with multiple safeguards:
1. **Semantic HMM initialization** - encourages Bearish state discovery
2. **Feature overlay** - backup short signal when TEMA<0 & vol>=80%
3. **Proper cost modeling** - realistic borrow fees
4. **Validation** - verify Bearish regime actually predicts poor returns

**Expected Behavior**:
- More complete market coverage (long/neutral/short)
- Better risk-adjusted returns in bear markets
- Higher strategy complexity (requires careful validation)
- Explicit cost tracking for transparency

ALLOCATION BY REGIME (NEW):
---------------------------
Bullish:  +100% SPY (full long exposure)
Neutral:   +50% SPY (reduced long exposure)
Bearish:   -50% SPY (SHORT exposure with borrow costs)

SHORT POSITION MECHANICS:
-------------------------
1. Signal generated at T-1 based on regime/features
2. Execute at Open[T]: Buy/Sell shares
3. For shorts (shares < 0):
   - Daily borrow cost = |shares| * Open[T] * borrow_rate_daily
   - Cost deducted from cash each day
4. Valuation at Open[T]: portfolio = cash + (shares * Open[T])
5. Returns: Open[T]/Open[T-1] - 1

VALIDATION REQUIREMENTS:
------------------------
Before using this strategy with real capital:

1. **Regime Validation**: 
   - Check regime vs forward returns matrix
   - Bearish regime should predict worse returns than Bullish
   - If not, regime detection is not useful

2. **Short Effectiveness**:
   - SPY should decline during short periods
   - Strategy should be negatively correlated with SPY when short
   - If not, timing is poor

3. **Cost Analysis**:
   - Verify borrow costs are reasonable
   - Check leverage limits are never exceeded
   - Ensure cash doesn't go persistently negative

4. **Comparison**:
   - Compare with long-only version
   - Shorts should improve risk-adjusted returns
   - If not, stick with simpler long/cash strategy

FEATURES:
---------
1. **TEMA** (Triple Exponential Moving Average): Trend indicator
2. **Yang-Zhang Volatility Percentiles**: Volatility regime indicator

EXECUTION FLOW:
--------------
Day T-1: Features from T-2, Regime detected, Signal generated
Day T: Execute at Open[T], Update positions, Calculate costs
Day T: Value at Open[T], Returns = Open[T]/Open[T-1] - 1

NO LOOK-AHEAD GUARANTEE:
-----------------------
- All features double-lagged (T-2 data for T predictions)
- HMM trained on expanding window only
- Feature overlay uses T-1 features for T signals
- Proper execution/attribution alignment
"""

from _market_data import download_ohlcv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from typing import Dict, Tuple, Optional, List
from sklearn.preprocessing import StandardScaler
from hmmlearn import hmm
from dataclasses import dataclass

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

class StrategyConfig:
    """
    Centralized configuration for all strategy parameters.
    
    DESIGN PRINCIPLE: All parameters in one place for easy modification.
    No hardcoded values scattered throughout the code.
    """
    
    # API Configuration
    
    # Date Range
    # CONFIGURABLE: Modify these for different backtest periods
    END_DATE = "2018-12-31"
    BACKTEST_YEARS = 5  # Will calculate start date automatically
    START_DATE = None  # Calculated in __post_init__
    
    # Asset Configuration
    TRADING_ASSET = "SPY"  # S&P 500 ETF
    BENCHMARK_ASSET = "URTH"  # Global benchmark
    
    # Capital and Costs
    INITIAL_CAPITAL = 100000.0
    
    # REALISTIC TRANSACTION COSTS
    # These reflect actual institutional trading costs
    COMMISSION_PER_TRADE = 0.0  # Many brokers now offer zero commission
    COMMISSION_PER_DOLLAR = 0.0001  # 1 basis point
    BID_ASK_SPREAD = 0.0002  # 2 basis points (SPY is very liquid)
    MARKET_IMPACT = 0.0001  # 1 basis point
    
    # Risk-Free Rate (used for cash returns)
    RISK_FREE_RATE_ANNUAL = 0.02  # 2% annual
    RISK_FREE_RATE_DAILY = 0.02 / 252
    
    # TEMA Configuration
    # TEMA = Triple Exponential Moving Average
    # Shorter period = more responsive, more signals
    # Longer period = smoother, fewer signals
    TEMA_PERIOD = 20  # 20-day TEMA (about 1 month)
    
    # Yang-Zhang Volatility Configuration
    # Window for volatility calculation
    # Shorter = more reactive to recent volatility
    # Longer = smoother volatility estimate
    YZ_WINDOW = 20  # 20-day window (about 1 month)
    
    # Percentile Configuration
    # IMPROVED: Rolling window instead of expanding to avoid crisis anchoring
    # Rolling window adapts to recent market conditions
    PERCENTILE_WINDOW = 252  # Use 1 year rolling window for percentiles
    MIN_PERCENTILE_HISTORY = 60  # At least 60 days for initial calculation
    
    # HMM Configuration
    # Number of hidden states (regimes)
    HMM_N_STATES = 3  # 3 regimes: Bullish, Neutral, Bearish
    HMM_N_ITER = 100  # Maximum iterations for EM algorithm
    HMM_RANDOM_STATE = 42  # For reproducibility
    HMM_COVARIANCE_TYPE = "full"  # Full covariance matrix
    
    # Walk-Forward Parameters
    # These control how often we refit the HMM
    # More frequent refitting = adapts faster but more computation
    MIN_TRAINING_DAYS = 252  # 1 year minimum before first prediction
    REFIT_FREQUENCY = 63  # Refit every quarter (63 trading days)
    
    # Regime Allocation
    # IMPROVED: Allow short positions in Bearish regime
    # How much to allocate to SPY in each regime
    # CONFIGURABLE: Adjust these based on risk tolerance
    BULLISH_ALLOCATION = 1.00   # 100% SPY (long)
    NEUTRAL_ALLOCATION = 0.50   # 50% SPY (reduced long)
    BEARISH_ALLOCATION = -0.50  # -50% SPY (SHORT position)
    
    # Short Position Controls
    # CRITICAL: Manage leverage and costs for short positions
    MAX_GROSS_LEVERAGE = 1.00   # Maximum absolute position size
    ENABLE_SHORTS = True  # Allow negative allocations
    BORROW_FEE_ANNUAL = 0.01  # 1% annual cost to borrow shares for shorting
    BORROW_FEE_DAILY = 0.01 / 252  # Daily borrow cost
    
    # Feature-Based Short Overlay
    # BACKUP: Force short signal when features indicate high risk
    # Even if HMM doesn't detect Bearish regime
    ENABLE_FEATURE_OVERLAY = True  # Enable feature-based override
    OVERLAY_TEMA_THRESHOLD = 0.0  # Short when TEMA_Normalized < 0 (downtrend)
    OVERLAY_VOL_THRESHOLD = 0.80  # AND volatility percentile >= 80% (high stress)
    
    # Rebalancing Configuration
    # IMPROVED: Event-driven rebalancing instead of fixed frequency
    REBALANCE_ON_REGIME_CHANGE = True  # Rebalance immediately on regime change
    REBALANCE_THRESHOLD = 0.05  # Rebalance if allocation deviates by >5%
    MIN_DAYS_BETWEEN_REBALANCE = 5  # Minimum 5 days between rebalances (reduce noise)
    
    # Visualization
    PLOT_WIDTH = 16
    PLOT_HEIGHT = 12
    
    def __post_init__(self):
        """Calculate derived parameters"""
        # Calculate start date based on backtest years
        end_dt = datetime.strptime(self.END_DATE, '%Y-%m-%d')
        start_dt = end_dt - timedelta(days=365 * self.BACKTEST_YEARS)
        self.START_DATE = start_dt.strftime('%Y-%m-%d')


# ============================================================================
# DATA ACQUISITION
# ============================================================================

class DataFetcher:
    """
    Handles data acquisition from Yahoo Finance.
    
    CRITICAL: No synthetic data, no forward/backward filling.
    We use ONLY the data that exists.
    """
    
    
    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str, buffer_days: int = 100) -> pd.DataFrame:
        buffer_start = pd.Timestamp(start_date) - pd.Timedelta(days=buffer_days)
        frame = download_ohlcv(ticker, buffer_start, end_date)
        if len(frame) < 100:
            raise ValueError(f"Only {len(frame)} rows for {ticker}; need at least 100.")
        return frame.rename_axis("Date").reset_index()[["Date", "Open", "High", "Low", "Close", "Volume"]]


# ============================================================================
# FEATURE ENGINEERING (NO LOOK-AHEAD)
# ============================================================================

class FeatureEngine:
    """
    Calculate TEMA and Yang-Zhang volatility percentiles WITHOUT look-ahead bias.
    
    CRITICAL PRINCIPLE: All features for date T use ONLY data up to T-1.
    Features are LAGGED to ensure T-1 availability at time T.
    """
    
    @staticmethod
    def calculate_ema(series: pd.Series, period: int) -> pd.Series:
        """
        Calculate Exponential Moving Average.
        
        EMA formula: EMA[t] = α * Price[t] + (1-α) * EMA[t-1]
        where α = 2/(period + 1)
        
        Parameters:
        -----------
        series : pd.Series
            Price series
        period : int
            EMA period
            
        Returns:
        --------
        pd.Series with EMA values
        """
        return series.ewm(span=period, adjust=False, min_periods=period).mean()
    
    @staticmethod
    def calculate_tema(close: pd.Series, period: int) -> pd.Series:
        """
        Calculate Triple Exponential Moving Average (TEMA).
        
        TEMA FORMULA:
        - EMA1 = EMA(Close, period)
        - EMA2 = EMA(EMA1, period)
        - EMA3 = EMA(EMA2, period)
        - TEMA = 3*EMA1 - 3*EMA2 + EMA3
        
        ADVANTAGES:
        - Reduces lag compared to simple moving averages
        - More responsive to recent price changes
        - Smoother than single EMA
        
        INTERPRETATION:
        - When Price > TEMA: Uptrend
        - When Price < TEMA: Downtrend
        - Crossovers indicate trend changes
        
        Parameters:
        -----------
        close : pd.Series
            Close price series
        period : int
            TEMA period
            
        Returns:
        --------
        pd.Series with TEMA values
        """
        # Calculate three levels of EMA
        ema1 = FeatureEngine.calculate_ema(close, period)
        ema2 = FeatureEngine.calculate_ema(ema1, period)
        ema3 = FeatureEngine.calculate_ema(ema2, period)
        
        # TEMA formula
        tema = 3 * ema1 - 3 * ema2 + ema3
        
        return tema
    
    @staticmethod
    def calculate_yang_zhang_volatility(df: pd.DataFrame, 
                                       window: int) -> pd.Series:
        """
        Calculate Yang-Zhang volatility estimator.
        
        YANG-ZHANG FORMULA:
        More efficient than close-to-close volatility because it uses
        intraday high/low information and overnight gaps.
        
        Components:
        1. Overnight volatility: log(Open[t] / Close[t-1])^2
        2. Open-to-close volatility: log(Close[t] / Open[t])^2
        3. High-low volatility: log(High / Low)^2
        
        Combined using Garman-Klass weights for optimal efficiency.
        
        ADVANTAGES:
        - More accurate than simple standard deviation
        - Uses all OHLC information
        - Better suited for assets with gaps (like stocks)
        
        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with Open, High, Low, Close columns
        window : int
            Rolling window for volatility calculation
            
        Returns:
        --------
        pd.Series with annualized Yang-Zhang volatility
        """
        # Validate OHLC relationships (data quality check)
        # Ensure: Low <= Open, Close <= High
        valid_mask = (
            (df["Low"] <= df["Open"]) & 
            (df["Open"] <= df["High"]) & 
            (df["Low"] <= df["Close"]) & 
            (df["Close"] <= df["High"])
        )
        
        if not valid_mask.all():
            print(f"WARNING: Found {(~valid_mask).sum()} invalid OHLC bars")
        
        # Calculate log returns components
        log_hl = np.log(df["High"] / df["Low"].replace(0, np.nan))
        log_co = np.log(df["Close"] / df["Open"].replace(0, np.nan))
        log_oc = np.log(df["Open"] / df["Close"].shift(1).replace(0, np.nan))
        
        # Rogers-Satchell component (intraday volatility)
        rs = log_hl * (log_hl - log_co) + log_co * log_co
        
        # Close-to-close volatility
        cc = log_co ** 2
        
        # Overnight volatility
        ov = log_oc ** 2
        
        # Garman-Klass coefficient (adjusts for window size)
        k = 0.34 / (1.34 + (window + 1) / (window - 1))
        
        # Rolling calculations (need at least half window of data)
        rs_roll = rs.rolling(window, min_periods=window//2).mean()
        cc_roll = cc.rolling(window, min_periods=window//2).mean()
        ov_roll = ov.rolling(window, min_periods=window//2).mean()
        
        # Yang-Zhang volatility (daily)
        yz_vol_daily = np.sqrt(ov_roll + k * cc_roll + (1 - k) * rs_roll)
        
        # Annualize (multiply by sqrt(252) trading days)
        yz_vol_annual = yz_vol_daily * np.sqrt(252)
        
        return yz_vol_annual
    
    @staticmethod
    def prepare_features(df: pd.DataFrame, config: StrategyConfig) -> pd.DataFrame:
        """
        Prepare all features with proper lagging to avoid look-ahead bias.
        
        ANTI-LOOK-AHEAD PROCESS:
        1. Calculate raw TEMA from close prices
        2. Calculate Yang-Zhang volatility from OHLC
        3. LAG both features by 1 day (critical step)
        4. Calculate percentile rankings using expanding window
        5. LAG percentiles again (double-lagging ensures T-1 to T)
        
        This ensures predictions for day T use ONLY information
        available at close of day T-1.
        
        Parameters:
        -----------
        df : pd.DataFrame
            OHLCV data
        config : StrategyConfig
            Configuration object with parameters
            
        Returns:
        --------
        pd.DataFrame with lagged features ready for HMM
        """
        result_df = df.copy()
        
        # ========== FEATURE 1: TEMA NORMALIZED ===========
        print(f"Calculating TEMA (period={config.TEMA_PERIOD})...")
        
        # Calculate TEMA
        tema = FeatureEngine.calculate_tema(
            result_df["Close"], 
            config.TEMA_PERIOD
        )
        
        # Normalize TEMA: (Price - TEMA) / TEMA
        # This gives us a percentage deviation from TEMA
        # Positive = Price above TEMA (uptrend)
        # Negative = Price below TEMA (downtrend)
        tema_normalized = (result_df["Close"] - tema) / tema
        
        # Store raw TEMA for reference
        result_df["TEMA"] = tema
        result_df["TEMA_Normalized"] = tema_normalized
        
        # CRITICAL: LAG TEMA by 1 day
        # At time T, we know TEMA from T-1
        result_df["TEMA_Normalized_Lag"] = tema_normalized.shift(1)
        
        print(f"  TEMA calculated and lagged")
        
        # ========== FEATURE 2: YANG-ZHANG VOLATILITY ===========
        print(f"Calculating Yang-Zhang volatility (window={config.YZ_WINDOW})...")
        
        # Calculate Yang-Zhang volatility
        yz_vol = FeatureEngine.calculate_yang_zhang_volatility(
            result_df, 
            config.YZ_WINDOW
        )
        
        # Store raw volatility
        result_df["YZ_Volatility"] = yz_vol
        
        # CRITICAL: LAG volatility by 1 day
        # At time T, we know volatility from T-1
        result_df["YZ_Volatility_Lag"] = yz_vol.shift(1)
        
        print(f"  Yang-Zhang volatility calculated and lagged")
        
        # ========== FEATURE 3: YANG-ZHANG PERCENTILE ===========
        print(f"Calculating Yang-Zhang percentiles (ROLLING window={config.PERCENTILE_WINDOW})...")
        
        # IMPROVED: Use ROLLING WINDOW instead of expanding
        # This prevents early crisis periods from anchoring percentiles forever
        # Rolling window adapts to recent market conditions
        
        # Initialize percentile series
        yz_percentile = pd.Series(np.nan, index=result_df.index)
        
        # Calculate percentiles using ROLLING window
        for i in range(config.MIN_PERCENTILE_HISTORY, len(result_df)):
            # ROLLING window: use last PERCENTILE_WINDOW observations
            # This adapts to recent market conditions
            window_start = max(0, i - config.PERCENTILE_WINDOW)
            hist_vol = result_df["YZ_Volatility_Lag"].iloc[window_start:i]
            
            # Remove NaN values
            hist_vol_clean = hist_vol.dropna()
            
            # Need minimum history for meaningful percentile
            if len(hist_vol_clean) >= config.MIN_PERCENTILE_HISTORY:
                # Current volatility (from previous day due to lag)
                current_vol = result_df["YZ_Volatility_Lag"].iloc[i]
                
                if pd.notna(current_vol):
                    # Calculate percentile rank
                    # What fraction of recent volatilities are below current?
                    percentile = (hist_vol_clean < current_vol).mean()
                    yz_percentile.iloc[i] = percentile
        
        result_df["YZ_Percentile"] = yz_percentile
        
        # CRITICAL: LAG percentile by 1 day (SECOND LAG)
        # At time T, we know percentile from T-1
        # This percentile was calculated using data up to T-2
        # So we have double-lagging: T-2 data -> T-1 percentile -> T prediction
        result_df["YZ_Percentile_Lag"] = yz_percentile.shift(1)
        
        print(f"  Yang-Zhang percentiles calculated with rolling window and lagged")
        
        # ========== CALCULATE RETURNS FOR REGIME LABELING ===========
        # Calculate returns for regime mapping (NOT used as features)
        # These are used only to determine which HMM state maps to which regime
        result_df["Returns"] = result_df["Close"].pct_change()
        
        print(f"  Features prepared: {result_df['TEMA_Normalized_Lag'].notna().sum()} "
              f"valid observations")
        
        return result_df


# ============================================================================
# WALK-FORWARD HMM REGIME DETECTOR
# ============================================================================

@dataclass
class RegimeMapping:
    """Stores regime mapping from training period"""
    state_to_regime: Dict[int, str]
    mean_returns: Dict[int, float]
    training_end_date: pd.Timestamp


class WalkForwardHMMDetector:
    """
    HMM Regime Detector with STRICT walk-forward validation.
    
    NO LOOK-AHEAD GUARANTEE:
    1. HMM trained only on expanding window of past data
    2. Features are double-lagged (T-2 data for T predictions)
    3. Regime labels assigned based on TRAINING PERIOD forward returns only
    4. Scaler and HMM model FROZEN during prediction window
    5. Periodic refitting as new data becomes available
    
    FEATURES USED:
    1. TEMA Normalized: Trend indicator
    2. Yang-Zhang Percentile: Volatility regime indicator
    
    Both features are lagged to ensure T-1 availability at time T.
    """
    
    def __init__(self, config: StrategyConfig):
        self.config = config
        
        # Features for HMM (both lagged)
        self.feature_cols = ["TEMA_Normalized_Lag", "YZ_Percentile_Lag"]
        
        # Regime names
        self.regime_names = ["Bullish", "Neutral", "Bearish"]
    
    def walk_forward_predict(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, List[Dict]]:
        """
        Perform walk-forward regime prediction with NO look-ahead bias.
        
        WALK-FORWARD PROCESS:
        --------------------
        For each prediction window:
        
        TRAINING PHASE (uses data up to T-1):
        1. Fit StandardScaler on training features
        2. Fit HMM on scaled features
        3. Create regime mapping using ONLY training forward returns
        
        PREDICTION PHASE (predict for day T):
        4. Transform day T features with FROZEN scaler
        5. Predict regime with FROZEN HMM
        6. Map state to regime label with FROZEN mapping
        
        ITERATION:
        7. Move forward one day
        8. Refit models every REFIT_FREQUENCY days
        9. Repeat until all days predicted
        
        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with calculated features
            
        Returns:
        --------
        Tuple of (predictions_df, fold_list)
        """
        print("\n" + "="*80)
        print("WALK-FORWARD HMM REGIME DETECTION (NO LOOK-AHEAD)")
        print("="*80)
        
        # Remove rows with missing features
        df_clean = df.dropna(subset=self.feature_cols).copy().reset_index(drop=True)
        
        print(f"Total observations: {len(df_clean)}")
        print(f"Date range: {df_clean['Date'].min().date()} to {df_clean['Date'].max().date()}")
        
        # Initialize result columns
        df_clean["Regime_State"] = np.nan
        df_clean["Regime"] = None
        
        # Walk-forward parameters
        min_train = self.config.MIN_TRAINING_DAYS
        refit_freq = self.config.REFIT_FREQUENCY
        
        print(f"\nWalk-forward configuration:")
        print(f"  Minimum training: {min_train} days (~{min_train/252:.1f} years)")
        print(f"  Refit frequency: {refit_freq} days (~{refit_freq/63:.1f} quarters)")
        print(f"  Predictions needed: {len(df_clean) - min_train}")
        
        # Track model fits
        folds_executed = []
        
        # Initialize model objects
        last_refit_idx = min_train
        scaler = None
        hmm_model = None
        regime_mapping = None
        
        fold_num = 0
        
        # WALK-FORWARD LOOP
        for i in range(min_train, len(df_clean)):
            days_since_refit = i - last_refit_idx
            
            # Check if we need to refit models
            if (scaler is None) or (days_since_refit >= refit_freq):
                fold_num += 1
                train_start = 0  # Expanding window (use all data up to this point)
                train_end = i    # Train up to current index (exclusive)
                
                print(f"\n{'='*80}")
                print(f"FOLD {fold_num}: Refitting models at index {i}")
                print(f"{'='*80}")
                print(f"Training window: indices [{train_start}:{train_end}]")
                print(f"Training dates: {df_clean.loc[train_start, 'Date'].date()} to "
                      f"{df_clean.loc[train_end-1, 'Date'].date()}")
                print(f"Training size: {train_end - train_start} days")
                
                # === TRAINING PHASE ===
                
                # STEP 1: Fit StandardScaler on training features
                X_train_raw = df_clean.loc[train_start:train_end-1, self.feature_cols].values
                
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_raw)
                print(f"StandardScaler fitted on {len(X_train_scaled)} observations")
                
                # STEP 2: Fit HMM with SEMANTIC INITIALIZATION
                # IMPROVED: Initialize HMM means based on feature quantiles
                # This encourages the HMM to discover all 3 regimes (Bearish, Neutral, Bullish)
                # even if one is rare in the training data
                
                # Calculate quantile-based initial means for each regime
                # Bearish: Low TEMA (25th percentile), High volatility (75th percentile)
                # Neutral: Median TEMA, Median volatility
                # Bullish: High TEMA (75th percentile), Low volatility (25th percentile)
                
                tema_train = X_train_scaled[:, 0]  # TEMA feature (standardized)
                yz_train = X_train_scaled[:, 1]     # YZ percentile feature (standardized)
                
                # Calculate quantiles
                tema_q25 = np.percentile(tema_train, 25)
                tema_q50 = np.percentile(tema_train, 50)
                tema_q75 = np.percentile(tema_train, 75)
                
                yz_q25 = np.percentile(yz_train, 25)
                yz_q50 = np.percentile(yz_train, 50)
                yz_q75 = np.percentile(yz_train, 75)
                
                # Initialize means semantically
                # Order: [Bearish, Neutral, Bullish]
                means_init = np.array([
                    [tema_q25, yz_q75],  # Bearish: Low trend, High vol
                    [tema_q50, yz_q50],  # Neutral: Medium trend, Medium vol
                    [tema_q75, yz_q25]   # Bullish: High trend, Low vol
                ])
                
                hmm_model = hmm.GaussianHMM(
                    n_components=self.config.HMM_N_STATES,
                    covariance_type=self.config.HMM_COVARIANCE_TYPE,
                    n_iter=self.config.HMM_N_ITER,
                    random_state=self.config.HMM_RANDOM_STATE,
                    means_prior=means_init  # Semantic initialization
                )
                
                # Set initial means
                hmm_model.means_ = means_init.copy()
                
                hmm_model.fit(X_train_scaled)
                print(f"HMM fitted with semantic initialization (converged: {hmm_model.monitor_.converged})")
                
                # STEP 3: Create regime mapping using FEATURE-BASED ANCHORING
                # IMPROVED: Stabilize labels using feature statistics instead of just forward returns
                # This prevents label switching across folds
                
                states_train = hmm_model.predict(X_train_scaled)
                
                # Calculate mean features for each state
                state_features = {}
                for state in range(self.config.HMM_N_STATES):
                    mask = states_train == state
                    if mask.sum() > 0:
                        state_mean_tema = X_train_scaled[mask, 0].mean()  # TEMA feature
                        state_mean_yz = X_train_scaled[mask, 1].mean()    # YZ percentile feature
                        state_features[state] = {
                            'tema': state_mean_tema,
                            'yz_percentile': state_mean_yz,
                            'count': mask.sum()
                        }
                    else:
                        state_features[state] = {'tema': 0, 'yz_percentile': 0.5, 'count': 0}
                
                # Get forward returns for validation
                returns_forward_train = df_clean.loc[train_start:train_end-1, "Returns"].shift(-1).values
                
                state_returns = {}
                for state in range(self.config.HMM_N_STATES):
                    mask = states_train == state
                    mask[-1] = False
                    if mask.sum() > 0:
                        state_returns[state] = returns_forward_train[mask].mean()
                    else:
                        state_returns[state] = 0
                
                # FEATURE-BASED REGIME ASSIGNMENT
                # High TEMA (uptrend) + Low volatility percentile (calm market) → Bullish
                # Low TEMA (downtrend) + High volatility percentile (stressed market) → Bearish
                # Middle → Neutral
                
                # Create scoring: higher score = more bullish
                state_scores = {}
                for state in range(self.config.HMM_N_STATES):
                    # Score based on features:
                    # Positive TEMA is bullish, high volatility percentile is bearish
                    score = state_features[state]['tema'] - state_features[state]['yz_percentile']
                    state_scores[state] = score
                
                # Sort by score: lowest → Bearish, middle → Neutral, highest → Bullish
                sorted_states = sorted(state_scores.items(), key=lambda x: x[1])
                
                state_to_regime = {
                    sorted_states[0][0]: "Bearish",   # Lowest score (low TEMA, high vol)
                    sorted_states[1][0]: "Neutral",   # Middle score
                    sorted_states[2][0]: "Bullish"    # Highest score (high TEMA, low vol)
                }
                
                # Store regime mapping
                regime_mapping = RegimeMapping(
                    state_to_regime=state_to_regime,
                    mean_returns=state_returns,
                    training_end_date=df_clean.loc[train_end-1, "Date"]
                )
                
                print(f"Regime mapping using FEATURE-BASED anchoring:")
                for state, regime in state_to_regime.items():
                    features = state_features[state]
                    print(f"  State {state} -> {regime:8s}: {features['count']:4d} obs, "
                          f"TEMA={features['tema']:+.3f}, YZ_pct={features['yz_percentile']:.3f}, "
                          f"fwd_ret={state_returns[state]:+.6f}")
                
                last_refit_idx = i
                
                # Track this fold
                folds_executed.append({
                    'fold': fold_num,
                    'train_start': train_start,
                    'train_end': train_end,
                    'train_start_date': df_clean.loc[train_start, 'Date'],
                    'train_end_date': df_clean.loc[train_end-1, 'Date'],
                    'regime_mapping': regime_mapping.state_to_regime.copy(),
                    'mean_returns': regime_mapping.mean_returns.copy()
                })
            
            # === PREDICTION PHASE ===
            
            # STEP 4: Transform current observation with FROZEN scaler
            X_current_raw = df_clean.loc[i:i, self.feature_cols].values
            X_current_scaled = scaler.transform(X_current_raw)
            
            # STEP 5: Predict regime with FROZEN HMM and FROZEN mapping
            state_pred = hmm_model.predict(X_current_scaled)[0]
            regime_pred = regime_mapping.state_to_regime[state_pred]
            
            df_clean.loc[i, "Regime_State"] = state_pred
            df_clean.loc[i, "Regime"] = regime_pred
        
        # === VALIDATION SUMMARY ===
        print(f"\n{'='*80}")
        print("WALK-FORWARD COMPLETE")
        print(f"{'='*80}")
        print(f"Total folds executed: {len(folds_executed)}")
        print(f"Predictions generated: {df_clean['Regime'].notna().sum()}")
        
        # Check regime distribution
        regime_counts = df_clean["Regime"].value_counts()
        print(f"\nRegime distribution:")
        for regime in self.regime_names:
            count = regime_counts.get(regime, 0)
            pct = count / len(df_clean) * 100 if len(df_clean) > 0 else 0
            print(f"  {regime:8s}: {count:4d} days ({pct:5.1f}%)")
        
        return df_clean, folds_executed


# ============================================================================
# BACKTESTING ENGINE
# ============================================================================

class BacktestEngine:
    """
    Realistic backtesting engine with proper execution flow.
    
    EXECUTION FLOW:
    --------------
    Day T-1 (Signal Day):
      - Observe Close[T-1]
      - Features from T-2 data (double-lagged)
      - Regime detected
      - Signal generated
      - Position sizing uses Close[T-1] value
    
    Day T (Execution Day):
      - Execute at Open[T]
      - Use Close[T-1] sizing (no look-ahead)
      - Portfolio positions updated
    
    Day T (Close):
      - Portfolio valued at Close[T]
      - Returns: Close[T]/Close[T-1] - 1
    """
    
    def __init__(self, config: StrategyConfig):
        self.config = config
        self.trades = []
    
    def calculate_trade_costs(self, trade_value: float) -> float:
        """
        Calculate realistic transaction costs.
        
        Components:
        - Commission per trade (if any)
        - Commission per dollar traded
        - Bid-ask spread (cost of liquidity)
        - Market impact (price movement from order)
        
        Parameters:
        -----------
        trade_value : float
            Dollar value of trade
            
        Returns:
        --------
        float : Total transaction cost
        """
        abs_value = abs(trade_value)
        
        # Fixed commission
        commission_fixed = self.config.COMMISSION_PER_TRADE
        
        # Variable commission
        commission_variable = abs_value * self.config.COMMISSION_PER_DOLLAR
        
        # Spread cost
        spread_cost = abs_value * self.config.BID_ASK_SPREAD
        
        # Market impact
        impact_cost = abs_value * self.config.MARKET_IMPACT
        
        total_cost = commission_fixed + commission_variable + spread_cost + impact_cost
        
        return total_cost
    
    def run_backtest(self, df_regimes: pd.DataFrame, 
                    df_spy: pd.DataFrame) -> Dict:
        """
        Execute backtest with realistic execution and costs.
        
        IMPROVED EXECUTION FLOW:
        -----------------------
        Day T-1 (Signal Day):
          - Portfolio valued at Close[T-1]
          - Features from T-2 data
          - Regime detected
          - Signal generated for T
        
        Day T (Execution Day):
          - Execute at Open[T] using T-1 signal
          - Portfolio positions updated
        
        Day T (End of Day):
          - Portfolio valued at Open[T]
          - Returns: Open[T]/Open[T-1] - 1 (ALIGNED WITH EXECUTION)
        
        IMPROVED REBALANCING:
        --------------------
        - Event-driven: Rebalance on regime change
        - Threshold-based: Rebalance if allocation drifts >5%
        - Minimum 5 days between rebalances to reduce noise
        
        Parameters:
        -----------
        df_regimes : pd.DataFrame
            Regime predictions
        df_spy : pd.DataFrame
            SPY price data
            
        Returns:
        --------
        Dict with comprehensive backtest results
        """
        print("\n" + "="*80)
        print("EXECUTING BACKTEST (IMPROVED)")
        print("="*80)
        
        # Merge regime predictions with SPY prices
        results = df_regimes[["Date", "Regime", "Regime_State"]].copy()
        
        # Add SPY prices
        spy_prices = df_spy[["Date", "Open", "Close"]].rename(
            columns={"Open": "SPY_Open", "Close": "SPY_Close"}
        )
        results = results.merge(spy_prices, on="Date", how="inner")
        results = results.reset_index(drop=True)
        
        print(f"Trading period: {len(results)} days")
        print(f"  Start: {results['Date'].min().date()}")
        print(f"  End: {results['Date'].max().date()}")
        
        # Initialize portfolio tracking
        results["Signal"] = 0.0  # Target allocation
        results["Position"] = 0.0  # Actual allocation
        results["Shares"] = 0.0
        results["Cash"] = 0.0
        results["Holdings"] = 0.0
        results["Portfolio_Value"] = 0.0
        results["Strategy_Returns"] = 0.0
        results["Trades"] = ""
        results["Prev_Regime"] = None  # Track regime changes
        
        # Allocation based on regime
        results.loc[results["Regime"] == "Bullish", "Signal"] = self.config.BULLISH_ALLOCATION
        results.loc[results["Regime"] == "Neutral", "Signal"] = self.config.NEUTRAL_ALLOCATION
        results.loc[results["Regime"] == "Bearish", "Signal"] = self.config.BEARISH_ALLOCATION
        
        # IMPROVED: Feature-based short overlay
        # Force short signal when features indicate high risk
        # Even if HMM doesn't detect Bearish regime
        if self.config.ENABLE_FEATURE_OVERLAY and self.config.ENABLE_SHORTS:
            # Merge regime data to get features
            feature_data = df_regimes[["Date", "TEMA_Normalized_Lag", "YZ_Percentile_Lag"]].copy()
            results = results.merge(feature_data, on="Date", how="left")
            
            # Identify high-risk conditions: Downtrend + High volatility
            overlay_condition = (
                (results["TEMA_Normalized_Lag"] < self.config.OVERLAY_TEMA_THRESHOLD) &
                (results["YZ_Percentile_Lag"] >= self.config.OVERLAY_VOL_THRESHOLD)
            )
            
            # Override signal with short allocation
            overlay_count = overlay_condition.sum()
            if overlay_count > 0:
                results.loc[overlay_condition, "Signal"] = min(
                    self.config.BEARISH_ALLOCATION, 0.0
                )
                print(f"  Feature overlay: {overlay_count} days forced to short")
        
        # IMPROVED: Apply leverage limits
        # Ensure no position exceeds maximum gross leverage
        if self.config.ENABLE_SHORTS:
            results["Signal"] = results["Signal"].clip(
                -self.config.MAX_GROSS_LEVERAGE,
                self.config.MAX_GROSS_LEVERAGE
            )
        
        print(f"\nAllocation by regime:")
        print(f"  Bullish: {self.config.BULLISH_ALLOCATION:.0%}")
        print(f"  Neutral: {self.config.NEUTRAL_ALLOCATION:.0%}")
        print(f"  Bearish: {self.config.BEARISH_ALLOCATION:.0%}")
        
        print(f"\nRebalancing rules:")
        print(f"  On regime change: {self.config.REBALANCE_ON_REGIME_CHANGE}")
        print(f"  Threshold: {self.config.REBALANCE_THRESHOLD:.1%}")
        print(f"  Min days between: {self.config.MIN_DAYS_BETWEEN_REBALANCE}")
        
        # Initialize portfolio state
        capital = self.config.INITIAL_CAPITAL
        cash = self.config.INITIAL_CAPITAL
        shares = 0.0
        
        # Initial valuation at Open[0]
        results.loc[0, "Cash"] = cash
        results.loc[0, "Portfolio_Value"] = cash
        results.loc[0, "Prev_Regime"] = results.loc[0, "Regime"]
        
        total_costs = 0.0
        last_rebalance_day = -100  # Allow immediate first rebalance
        rebalance_count = 0
        regime_change_rebalances = 0
        threshold_rebalances = 0
        
        print(f"\nStarting backtest with ${capital:,.0f} capital...")
        
        # Execute backtest day by day
        for i in range(1, len(results)):
            current_regime = results.loc[i, "Regime"]
            prev_regime = results.loc[i-1, "Regime"]
            
            # Track regime changes
            results.loc[i, "Prev_Regime"] = prev_regime
            
            # Get desired allocation from PREVIOUS day's signal
            desired_alloc = results.loc[i-1, "Signal"]
            
            # Calculate current allocation based on Open[i] prices
            current_open = results.loc[i, "SPY_Open"]
            if pd.notna(current_open) and current_open > 0:
                current_holdings_value = shares * current_open
                prev_portfolio_value = cash + current_holdings_value
                current_alloc = current_holdings_value / prev_portfolio_value if prev_portfolio_value > 0 else 0
            else:
                current_alloc = 0
                current_holdings_value = 0
                prev_portfolio_value = cash
            
            # === REBALANCING DECISION LOGIC ===
            days_since_rebal = i - last_rebalance_day
            
            # Check rebalancing triggers
            regime_changed = (current_regime != prev_regime) if pd.notna(current_regime) and pd.notna(prev_regime) else False
            allocation_drift = abs(desired_alloc - current_alloc)
            sufficient_time = days_since_rebal >= self.config.MIN_DAYS_BETWEEN_REBALANCE
            
            should_rebalance = False
            rebalance_reason = ""
            
            if sufficient_time:
                if self.config.REBALANCE_ON_REGIME_CHANGE and regime_changed:
                    should_rebalance = True
                    rebalance_reason = "REGIME_CHANGE"
                    regime_change_rebalances += 1
                elif allocation_drift > self.config.REBALANCE_THRESHOLD:
                    should_rebalance = True
                    rebalance_reason = "THRESHOLD"
                    threshold_rebalances += 1
            
            # Track if trades executed
            trades_today = []
            
            # === REBALANCING EXECUTION ===
            if should_rebalance and pd.notna(current_open) and current_open > 0:
                # Calculate target value
                target_value = desired_alloc * prev_portfolio_value
                current_value = shares * current_open
                trade_value = target_value - current_value
                
                if abs(trade_value) > prev_portfolio_value * 0.001:  # Only if >0.1% of portfolio
                    # Calculate shares to trade
                    shares_to_trade = trade_value / current_open
                    
                    # Calculate costs
                    cost = self.calculate_trade_costs(abs(trade_value))
                    total_costs += cost
                    
                    # Update position
                    shares += shares_to_trade
                    cash -= trade_value + cost
                    
                    # Record trade
                    action = "BUY" if shares_to_trade > 0 else "SELL"
                    trades_today.append(f"{action}_{rebalance_reason}")
                    
                    self.trades.append({
                        "Date": results.loc[i, "Date"],
                        "Action": action,
                        "Reason": rebalance_reason,
                        "Shares": abs(shares_to_trade),
                        "Price": current_open,
                        "Value": abs(trade_value),
                        "Cost": cost,
                        "Regime": current_regime,
                        "Prev_Regime": prev_regime
                    })
                    
                    last_rebalance_day = i
                    rebalance_count += 1
            
            # === PORTFOLIO VALUATION ===
            # IMPROVED: Use Open[T] for valuation (aligned with execution)
            
            # Calculate cash interest (risk-free rate)
            # CRITICAL: Only positive cash earns interest
            # Negative cash (margin debt) does NOT earn interest here
            # (borrow costs handled separately below)
            if i > 0 and cash > 0:
                cash += cash * self.config.RISK_FREE_RATE_DAILY
            
            # Calculate borrow costs for short positions
            # IMPROVED: When shares < 0 (short), we pay daily borrow fee
            if self.config.ENABLE_SHORTS and shares < 0:
                # Borrow cost = |notional value| * daily borrow rate
                # Notional value = |shares| * current price
                borrow_notional = abs(shares) * current_open
                borrow_cost = borrow_notional * self.config.BORROW_FEE_DAILY
                cash -= borrow_cost  # Subtract borrow cost from cash
                
                # Track borrow costs
                if i == 1 or "Borrow_Cost" not in results.columns:
                    results["Borrow_Cost"] = 0.0
                results.loc[i, "Borrow_Cost"] = borrow_cost
            
            # Value holdings at Open[T]
            # CRITICAL: For shorts (shares < 0), holdings_value will be negative
            # This is correct: short position has negative value
            if pd.notna(current_open):
                holdings_value = shares * current_open
            else:
                holdings_value = 0
            
            portfolio_value = cash + holdings_value
            
            # IMPROVED: Calculate Open-to-Open returns (ALIGNED WITH EXECUTION)
            # Returns from Open[T-1] to Open[T]
            if i > 0:
                prev_open = results.loc[i-1, "SPY_Open"]
                if pd.notna(current_open) and pd.notna(prev_open) and prev_open > 0:
                    prev_portfolio = results.loc[i-1, "Portfolio_Value"]
                    if prev_portfolio > 0:
                        results.loc[i, "Strategy_Returns"] = portfolio_value / prev_portfolio - 1
            
            # Update state
            results.loc[i, "Shares"] = shares
            results.loc[i, "Cash"] = cash
            results.loc[i, "Holdings"] = holdings_value
            results.loc[i, "Portfolio_Value"] = portfolio_value
            results.loc[i, "Position"] = holdings_value / portfolio_value if portfolio_value > 0 else 0
            results.loc[i, "Trades"] = ", ".join(trades_today) if trades_today else ""
            
            # Progress update
            if i % 250 == 0:
                print(f"  Day {i}/{len(results)}: Portfolio = ${portfolio_value:,.0f}, "
                      f"Regime = {current_regime}")
        
        # Calculate cumulative returns
        results["Cumulative_Returns"] = (1 + results["Strategy_Returns"]).cumprod()
        
        # Calculate short position statistics
        short_days = (results["Position"] < -0.01).sum()
        total_borrow_cost = results["Borrow_Cost"].sum() if "Borrow_Cost" in results.columns else 0
        
        print(f"\nBacktest complete:")
        print(f"  Total rebalances: {rebalance_count}")
        print(f"    Regime change: {regime_change_rebalances}")
        print(f"    Threshold drift: {threshold_rebalances}")
        print(f"  Total costs: ${total_costs:,.2f}")
        print(f"  Short position days: {short_days}")
        print(f"  Total borrow costs: ${total_borrow_cost:,.2f}")
        print(f"  Final value: ${results.loc[len(results)-1, 'Portfolio_Value']:,.2f}")
        
        # Validate leverage limits
        max_gross_position = results["Position"].abs().max()
        if max_gross_position > self.config.MAX_GROSS_LEVERAGE + 0.01:
            print(f"  WARNING: Max gross leverage {max_gross_position:.2%} exceeds limit!")
        
        return results, total_costs
    
    def add_benchmark(self, results: pd.DataFrame, df_benchmark: pd.DataFrame) -> pd.DataFrame:
        """
        Add buy-and-hold benchmark for comparison.
        
        IMPROVED: Use Open-to-Open returns for fair comparison with strategy
        
        Parameters:
        -----------
        results : pd.DataFrame
            Strategy results
        df_benchmark : pd.DataFrame
            Benchmark price data
            
        Returns:
        --------
        pd.DataFrame with benchmark added
        """
        print(f"\nAdding {self.config.BENCHMARK_ASSET} benchmark (Open-to-Open returns)...")
        
        # Merge benchmark prices
        bench_prices = df_benchmark[["Date", "Open", "Close"]].rename(
            columns={"Open": "Benchmark_Open", "Close": "Benchmark_Close"}
        )
        results = results.merge(bench_prices, on="Date", how="left")
        
        # IMPROVED: Calculate Open-to-Open returns (same as strategy)
        initial_open = results.loc[0, "Benchmark_Open"]
        if pd.notna(initial_open) and initial_open > 0:
            # Buy-and-hold: invest all capital at Open[0]
            initial_cost = self.calculate_trade_costs(self.config.INITIAL_CAPITAL)
            shares_bench = (self.config.INITIAL_CAPITAL - initial_cost) / initial_open
            
            # Value at each day's Open (aligned with strategy)
            results["Benchmark_Value"] = shares_bench * results["Benchmark_Open"]
            
            # Calculate Open-to-Open returns
            results["Benchmark_Returns"] = results["Benchmark_Open"].pct_change()
            results["Benchmark_Cumulative"] = (1 + results["Benchmark_Returns"]).cumprod()
            
            print(f"  Benchmark added: ${results['Benchmark_Value'].iloc[-1]:,.2f}")
            print(f"  Benchmark uses Open-to-Open returns (aligned with strategy)")
        
        return results


# ============================================================================
# VALIDATION AND DIAGNOSTICS
# ============================================================================

class ValidationDiagnostics:
    """
    Validation tools to ensure strategy quality.
    
    CRITICAL VALIDATIONS:
    1. Regime vs Forward Returns matrix
    2. Leverage limit compliance
    3. Borrow cost tracking
    4. Short position profitability
    """
    
    @staticmethod
    def create_regime_forward_returns_matrix(results: pd.DataFrame, 
                                            forward_days: List[int] = [1, 3, 5, 10, 21]) -> pd.DataFrame:
        """
        Create matrix of regime vs forward returns.
        
        CRITICAL VALIDATION: Verify that "Bearish" regime actually
        predicts worse forward returns than other regimes.
        
        If Bearish regime shows positive forward returns similar to
        Bullish, then the regime detection is not useful.
        
        Parameters:
        -----------
        results : pd.DataFrame
            Backtest results with regime labels
        forward_days : List[int]
            List of forward-looking periods to analyze
            
        Returns:
        --------
        pd.DataFrame with regime vs forward return statistics
        """
        print("\n" + "="*80)
        print("REGIME VS FORWARD RETURNS VALIDATION")
        print("="*80)
        
        validation_results = []
        
        for days in forward_days:
            # Calculate forward returns
            fwd_returns = results["SPY_Open"].pct_change(days).shift(-days)
            
            # Group by regime
            regime_stats = results.groupby("Regime").apply(
                lambda x: pd.Series({
                    'count': len(x),
                    'mean_fwd_return': fwd_returns.loc[x.index].mean(),
                    'median_fwd_return': fwd_returns.loc[x.index].median(),
                    'std_fwd_return': fwd_returns.loc[x.index].std(),
                    'sharpe_fwd': (fwd_returns.loc[x.index].mean() / fwd_returns.loc[x.index].std()) if fwd_returns.loc[x.index].std() > 0 else 0
                })
            )
            
            regime_stats['forward_days'] = days
            validation_results.append(regime_stats)
        
        # Combine all periods
        validation_df = pd.concat(validation_results, keys=[f"{d}d" for d in forward_days])
        
        print("\nRegime Forward Return Statistics:")
        print(validation_df.to_string())
        
        # Check if Bearish regime has worse returns
        print("\n" + "-"*80)
        print("VALIDATION CHECKS:")
        print("-"*80)
        
        for days in forward_days:
            key = f"{days}d"
            if key in validation_df.index:
                stats = validation_df.loc[key]
                
                if "Bearish" in stats.index and "Bullish" in stats.index:
                    bearish_ret = stats.loc["Bearish", "mean_fwd_return"]
                    bullish_ret = stats.loc["Bullish", "mean_fwd_return"]
                    
                    if bearish_ret < bullish_ret:
                        print(f"✓ {days}d forward: Bearish ({bearish_ret:.4f}) < Bullish ({bullish_ret:.4f})")
                    else:
                        print(f"✗ {days}d forward: Bearish ({bearish_ret:.4f}) >= Bullish ({bullish_ret:.4f}) - REGIME NOT PREDICTIVE!")
                elif "Bearish" not in stats.index:
                    print(f"! {days}d forward: NO BEARISH REGIME DETECTED")
        
        return validation_df
    
    @staticmethod
    def analyze_short_performance(results: pd.DataFrame) -> Dict:
        """
        Analyze performance of short positions.
        
        Returns:
        --------
        Dict with short position statistics
        """
        print("\n" + "="*80)
        print("SHORT POSITION ANALYSIS")
        print("="*80)
        
        # Identify short periods (Position < 0)
        short_mask = results["Position"] < -0.01
        short_days = short_mask.sum()
        
        if short_days == 0:
            print("No short positions taken during backtest period")
            return {}
        
        # Calculate short performance
        short_returns = results.loc[short_mask, "Strategy_Returns"]
        short_spy_returns = results.loc[short_mask, "SPY_Open"].pct_change()
        
        stats = {
            'short_days': short_days,
            'short_pct_of_time': short_days / len(results) * 100,
            'avg_short_return': short_returns.mean(),
            'median_short_return': short_returns.median(),
            'total_short_return': (1 + short_returns).prod() - 1,
            'win_rate': (short_returns > 0).sum() / len(short_returns) * 100 if len(short_returns) > 0 else 0,
            'avg_spy_return_during_shorts': short_spy_returns.mean(),
            'correlation_spy': short_returns.corr(short_spy_returns)
        }
        
        print(f"\nShort Position Statistics:")
        print(f"  Days short: {stats['short_days']} ({stats['short_pct_of_time']:.1f}% of time)")
        print(f"  Average daily return: {stats['avg_short_return']:.4f}")
        print(f"  Total return from shorts: {stats['total_short_return']:.2%}")
        print(f"  Win rate: {stats['win_rate']:.1f}%")
        print(f"  SPY avg return during shorts: {stats['avg_spy_return_during_shorts']:.4f}")
        print(f"  Correlation with SPY: {stats['correlation_spy']:.3f}")
        
        # Validate short effectiveness
        print("\n" + "-"*80)
        print("SHORT EFFECTIVENESS VALIDATION:")
        print("-"*80)
        
        if stats['avg_spy_return_during_shorts'] < 0:
            print(f"✓ SPY declined during short periods (avg: {stats['avg_spy_return_during_shorts']:.4f})")
        else:
            print(f"✗ SPY rose during short periods (avg: {stats['avg_spy_return_during_shorts']:.4f}) - POOR TIMING!")
        
        if stats['correlation_spy'] < -0.3:
            print(f"✓ Strategy negatively correlated with SPY during shorts ({stats['correlation_spy']:.3f})")
        else:
            print(f"! Strategy correlation with SPY during shorts: {stats['correlation_spy']:.3f}")
        
        return stats


# ============================================================================
# PERFORMANCE METRICS
# ============================================================================

class PerformanceMetrics:
    """Calculate comprehensive performance metrics"""
    
    @staticmethod
    def calculate_all_metrics(results: pd.DataFrame, 
                             config: StrategyConfig) -> Dict:
        """
        Calculate performance metrics for strategy and benchmark.
        
        Metrics:
        - Total return and CAGR
        - Volatility (annualized)
        - Sharpe ratio
        - Maximum drawdown
        - Win rate
        """
        metrics = {}
        
        # Strategy metrics
        strategy_returns = results["Strategy_Returns"].dropna()
        n_years = len(strategy_returns) / 252
        
        final_value = results["Portfolio_Value"].iloc[-1]
        initial_value = config.INITIAL_CAPITAL
        
        total_return = (final_value / initial_value) - 1
        cagr = ((final_value / initial_value) ** (1/n_years) - 1) if n_years > 0 else 0
        volatility = strategy_returns.std() * np.sqrt(252)
        sharpe = np.sqrt(252) * (strategy_returns.mean() - config.RISK_FREE_RATE_DAILY) / strategy_returns.std() if strategy_returns.std() > 0 else 0
        
        # Drawdown
        cumulative = (1 + strategy_returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min()
        
        # Win rate
        win_rate = (strategy_returns > 0).sum() / len(strategy_returns) if len(strategy_returns) > 0 else 0
        
        metrics["Strategy"] = {
            "Total Return": total_return,
            "CAGR": cagr,
            "Volatility": volatility,
            "Sharpe Ratio": sharpe,
            "Max Drawdown": max_dd,
            "Win Rate": win_rate,
            "Final Value": final_value
        }
        
        # Benchmark metrics
        if "Benchmark_Returns" in results.columns:
            bench_returns = results["Benchmark_Returns"].dropna()
            bench_final = results["Benchmark_Value"].iloc[-1]
            
            bench_total = (bench_final / initial_value) - 1
            bench_cagr = ((bench_final / initial_value) ** (1/n_years) - 1) if n_years > 0 else 0
            bench_vol = bench_returns.std() * np.sqrt(252)
            bench_sharpe = np.sqrt(252) * (bench_returns.mean() - config.RISK_FREE_RATE_DAILY) / bench_returns.std() if bench_returns.std() > 0 else 0
            
            bench_cum = (1 + bench_returns).cumprod()
            bench_max = bench_cum.expanding().max()
            bench_dd = (bench_cum - bench_max) / bench_max
            bench_max_dd = bench_dd.min()
            
            metrics["Benchmark"] = {
                "Total Return": bench_total,
                "CAGR": bench_cagr,
                "Volatility": bench_vol,
                "Sharpe Ratio": bench_sharpe,
                "Max Drawdown": bench_max_dd,
                "Final Value": bench_final
            }
        
        return metrics
    
    @staticmethod
    def print_metrics(metrics: Dict):
        """Print formatted metrics table"""
        print("\n" + "="*80)
        print("PERFORMANCE METRICS")
        print("="*80)
        
        for name, m in metrics.items():
            print(f"\n{name}:")
            print(f"  Total Return:     {m['Total Return']:>10.2%}")
            print(f"  CAGR:             {m['CAGR']:>10.2%}")
            print(f"  Volatility:       {m['Volatility']:>10.2%}")
            print(f"  Sharpe Ratio:     {m['Sharpe Ratio']:>10.2f}")
            print(f"  Max Drawdown:     {m['Max Drawdown']:>10.2%}")
            if 'Win Rate' in m:
                print(f"  Win Rate:         {m['Win Rate']:>10.2%}")
            print(f"  Final Value:      ${m['Final Value']:>10,.2f}")
        
        if "Strategy" in metrics and "Benchmark" in metrics:
            print(f"\nRelative Performance:")
            print(f"  Excess Return:    {metrics['Strategy']['Total Return'] - metrics['Benchmark']['Total Return']:>10.2%}")
            print(f"  Excess CAGR:      {metrics['Strategy']['CAGR'] - metrics['Benchmark']['CAGR']:>10.2%}")
        
        print("="*80)


# ============================================================================
# VISUALIZATION
# ============================================================================

def create_visualizations(results: pd.DataFrame, config: StrategyConfig):
    """
    Create comprehensive visualizations with improvements.
    
    IMPROVEMENTS:
    - Cumulative returns start from 0% (not 100)
    - Show excess returns over risk-free rate
    - Mark regime transitions
    - Better regime distribution visualization
    
    NO HARDCODING: All parameters from config
    """
    fig, axes = plt.subplots(4, 2, figsize=(config.PLOT_WIDTH, config.PLOT_HEIGHT))
    
    # Colors (configurable)
    regime_colors = {
        "Bullish": "green",
        "Neutral": "gray",
        "Bearish": "red"
    }
    
    # Plot 1: Portfolio Performance
    ax1 = axes[0, 0]
    ax1.plot(results["Date"], results["Portfolio_Value"],
            label="Strategy", linewidth=2, color="blue")
    if "Benchmark_Value" in results.columns:
        ax1.plot(results["Date"], results["Benchmark_Value"],
                label=f"{config.BENCHMARK_ASSET} (Benchmark)", 
                linewidth=2, alpha=0.7, color="orange")
    ax1.set_title("Portfolio Value", fontsize=12, fontweight="bold")
    ax1.set_ylabel("Value ($)", fontsize=10)
    ax1.legend(fontsize=9)
    ax1.grid(alpha=0.3)
    ax1.axhline(y=config.INITIAL_CAPITAL, color='black', 
                linestyle='--', alpha=0.3, linewidth=1)
    
    # Plot 2: Regime Detection with Transitions
    ax2 = axes[0, 1]
    
    # Plot regime transitions as vertical lines
    regime_changes = results[results["Regime"] != results["Prev_Regime"]].index
    for idx in regime_changes[1:]:  # Skip first (no previous regime)
        ax2.axvline(x=results.loc[idx, "Date"], color='purple', 
                   alpha=0.2, linestyle='--', linewidth=0.8)
    
    # Plot regime scatter
    for regime, color in regime_colors.items():
        mask = results["Regime"] == regime
        if mask.sum() > 0:
            ax2.scatter(results.loc[mask, "Date"],
                       results.loc[mask, "SPY_Close"],
                       c=color, label=regime, alpha=0.6, s=10)
    
    ax2.set_title("Regime Detection with Transitions", fontsize=12, fontweight="bold")
    ax2.set_ylabel("SPY Price ($)", fontsize=10)
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)
    
    # Plot 3: Cumulative Returns (IMPROVED: Start from 0%)
    ax3 = axes[1, 0]
    # IMPROVEMENT: Show as percentage return from starting capital (0% baseline)
    strategy_cum_pct = (results["Cumulative_Returns"] - 1) * 100
    ax3.plot(results["Date"], strategy_cum_pct,
            label="Strategy", linewidth=2, color="blue")
    if "Benchmark_Cumulative" in results.columns:
        bench_cum_pct = (results["Benchmark_Cumulative"] - 1) * 100
        ax3.plot(results["Date"], bench_cum_pct,
                label=f"{config.BENCHMARK_ASSET}", linewidth=2, 
                alpha=0.7, color="orange")
    ax3.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
    ax3.set_title("Cumulative Returns (from 0% baseline)", fontsize=12, fontweight="bold")
    ax3.set_ylabel("Return (%)", fontsize=10)
    ax3.legend(fontsize=9)
    ax3.grid(alpha=0.3)
    
    # Plot 4: Excess Returns over Risk-Free Rate (NEW)
    ax4 = axes[1, 1]
    # Calculate cumulative excess returns
    strategy_excess = results["Strategy_Returns"] - config.RISK_FREE_RATE_DAILY
    strategy_excess_cum = (1 + strategy_excess).cumprod()
    ax4.plot(results["Date"], (strategy_excess_cum - 1) * 100,
            linewidth=2, color="darkblue", label="Strategy Excess")
    if "Benchmark_Returns" in results.columns:
        bench_excess = results["Benchmark_Returns"] - config.RISK_FREE_RATE_DAILY
        bench_excess_cum = (1 + bench_excess).cumprod()
        ax4.plot(results["Date"], (bench_excess_cum - 1) * 100,
                linewidth=2, color="darkorange", alpha=0.7, 
                label=f"{config.BENCHMARK_ASSET} Excess")
    ax4.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
    ax4.set_title("Excess Returns over Risk-Free Rate", fontsize=12, fontweight="bold")
    ax4.set_ylabel("Excess Return (%)", fontsize=10)
    ax4.legend(fontsize=9)
    ax4.grid(alpha=0.3)
    
    # Plot 5: Drawdowns
    ax5 = axes[2, 0]
    strategy_cum = results["Cumulative_Returns"]
    strategy_dd = (strategy_cum / strategy_cum.expanding().max() - 1) * 100
    ax5.fill_between(results["Date"], strategy_dd, 0,
                     alpha=0.5, color="red", label="Strategy")
    if "Benchmark_Cumulative" in results.columns:
        bench_cum = results["Benchmark_Cumulative"]
        bench_dd = (bench_cum / bench_cum.expanding().max() - 1) * 100
        ax5.plot(results["Date"], bench_dd, color="orange", 
                alpha=0.7, linewidth=1.5, label="Benchmark")
    ax5.set_title("Drawdown Comparison", fontsize=12, fontweight="bold")
    ax5.set_ylabel("Drawdown (%)", fontsize=10)
    ax5.legend(fontsize=9)
    ax5.grid(alpha=0.3)
    
    # Plot 6: Position Allocation with Trades (IMPROVED: Show shorts)
    ax6 = axes[2, 1]
    
    # Separate long and short positions for visualization
    long_positions = results["Position"].clip(lower=0) * 100
    short_positions = results["Position"].clip(upper=0) * 100
    
    # Plot long positions (green)
    ax6.fill_between(results["Date"], 0, long_positions,
                     color="green", alpha=0.3, label="Long")
    ax6.plot(results["Date"], long_positions,
            color="green", linewidth=2, alpha=0.7)
    
    # Plot short positions (red)
    ax6.fill_between(results["Date"], 0, short_positions,
                     color="red", alpha=0.3, label="Short")
    ax6.plot(results["Date"], short_positions,
            color="red", linewidth=2, alpha=0.7)
    
    # Mark trade days
    trade_days = results[results["Trades"] != ""]
    if len(trade_days) > 0:
        ax6.scatter(trade_days["Date"], trade_days["Position"] * 100,
                   color="purple", s=30, alpha=0.6, marker='o', 
                   label=f"Rebalances ({len(trade_days)})")
    
    ax6.axhline(y=0, color='black', linestyle='-', alpha=0.5, linewidth=1)
    ax6.set_title("SPY Allocation with Shorts (IMPROVED)", fontsize=12, fontweight="bold")
    ax6.set_ylabel("Allocation (%)", fontsize=10)
    ax6.legend(fontsize=9)
    ax6.grid(alpha=0.3)
    
    # Plot 7: Regime Distribution (IMPROVED)
    ax7 = axes[3, 0]
    regime_counts = results["Regime"].value_counts()
    total_days = len(results)
    colors = [regime_colors[r] for r in regime_counts.index]
    bars = ax7.bar(regime_counts.index, regime_counts.values, 
                   color=colors, alpha=0.7)
    
    # Add percentage labels on bars
    for bar, regime in zip(bars, regime_counts.index):
        height = bar.get_height()
        count = regime_counts[regime]
        pct = count / total_days * 100
        ax7.text(bar.get_x() + bar.get_width()/2., height,
                f'{count}\n({pct:.1f}%)',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax7.set_title("Regime Distribution", fontsize=12, fontweight="bold")
    ax7.set_ylabel("Number of Days", fontsize=10)
    ax7.set_xlabel("Regime", fontsize=10)
    ax7.grid(alpha=0.3, axis="y")
    
    # Plot 8: Rolling Sharpe Ratio (NEW)
    ax8 = axes[3, 1]
    rolling_window = 63  # 3-month rolling window
    strategy_rolling_mean = results["Strategy_Returns"].rolling(rolling_window).mean() * 252
    strategy_rolling_std = results["Strategy_Returns"].rolling(rolling_window).std() * np.sqrt(252)
    strategy_rolling_sharpe = (strategy_rolling_mean - config.RISK_FREE_RATE_ANNUAL) / strategy_rolling_std
    
    ax8.plot(results["Date"], strategy_rolling_sharpe,
            linewidth=2, color="darkblue", label="Strategy")
    
    if "Benchmark_Returns" in results.columns:
        bench_rolling_mean = results["Benchmark_Returns"].rolling(rolling_window).mean() * 252
        bench_rolling_std = results["Benchmark_Returns"].rolling(rolling_window).std() * np.sqrt(252)
        bench_rolling_sharpe = (bench_rolling_mean - config.RISK_FREE_RATE_ANNUAL) / bench_rolling_std
        ax8.plot(results["Date"], bench_rolling_sharpe,
                linewidth=2, color="darkorange", alpha=0.7,
                label=config.BENCHMARK_ASSET)
    
    ax8.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
    ax8.axhline(y=1, color='green', linestyle='--', alpha=0.3, linewidth=1, label="Sharpe=1")
    ax8.set_title(f"Rolling Sharpe Ratio ({rolling_window}-day)", 
                  fontsize=12, fontweight="bold")
    ax8.set_ylabel("Sharpe Ratio", fontsize=10)
    ax8.set_xlabel("Date", fontsize=10)
    ax8.legend(fontsize=9)
    ax8.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function"""
    
    print("="*80)
    print("HMM STRATEGY: TEMA + YANG-ZHANG VOLATILITY PERCENTILES")
    print("="*80)
    
    # Initialize configuration
    config = StrategyConfig()
    config.__post_init__()
    
    print(f"\nConfiguration:")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  Asset: {config.TRADING_ASSET}")
    print(f"  Benchmark: {config.BENCHMARK_ASSET}")
    print(f"  Initial Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  TEMA Period: {config.TEMA_PERIOD}")
    print(f"  Yang-Zhang Window: {config.YZ_WINDOW}")
    
    # ===== DATA ACQUISITION =====
    print("\n" + "-"*80)
    print("DATA ACQUISITION")
    print("-"*80)
    
    fetcher = DataFetcher()
    
    # Fetch SPY data (with buffer for indicators)
    df_spy = fetcher.fetch_historical_data(
        config.TRADING_ASSET,
        config.START_DATE,
        config.END_DATE,
        buffer_days=config.MIN_TRAINING_DAYS + 100
    )
    
    # Fetch benchmark
    df_benchmark = fetcher.fetch_historical_data(
        config.BENCHMARK_ASSET,
        config.START_DATE,
        config.END_DATE,
        buffer_days=100
    )
    
    # ===== FEATURE ENGINEERING =====
    print("\n" + "-"*80)
    print("FEATURE ENGINEERING")
    print("-"*80)
    
    df_features = FeatureEngine.prepare_features(df_spy, config)
    
    # ===== REGIME DETECTION =====
    print("\n" + "-"*80)
    print("REGIME DETECTION")
    print("-"*80)
    
    detector = WalkForwardHMMDetector(config)
    df_regimes, folds = detector.walk_forward_predict(df_features)
    
    # ===== BACKTESTING =====
    print("\n" + "-"*80)
    print("BACKTESTING")
    print("-"*80)
    
    engine = BacktestEngine(config)
    results, total_costs = engine.run_backtest(df_regimes, df_spy)
    
    # Add benchmark
    results = engine.add_benchmark(results, df_benchmark)
    
    # ===== PERFORMANCE ANALYSIS =====
    print("\n" + "-"*80)
    print("PERFORMANCE ANALYSIS")
    print("-"*80)
    
    metrics = PerformanceMetrics.calculate_all_metrics(results, config)
    PerformanceMetrics.print_metrics(metrics)
    
    # ===== VALIDATION DIAGNOSTICS (NEW) =====
    print("\n" + "-"*80)
    print("VALIDATION DIAGNOSTICS")
    print("-"*80)
    
    # Validate regime predictiveness
    validation_matrix = ValidationDiagnostics.create_regime_forward_returns_matrix(
        results, 
        forward_days=[1, 3, 5, 10, 21]
    )
    
    # Analyze short performance (if shorts enabled)
    if config.ENABLE_SHORTS:
        short_stats = ValidationDiagnostics.analyze_short_performance(results)
    
    # ===== VISUALIZATIONS =====
    print("\n" + "-"*80)
    print("GENERATING VISUALIZATIONS")
    print("-"*80)
    
    create_visualizations(results, config)
    
    # ===== EXPORT RESULTS =====
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    results.to_csv(f"hmm_tema_yz_results_{timestamp}.csv", index=False)
    pd.DataFrame(engine.trades).to_csv(f"hmm_tema_yz_trades_{timestamp}.csv", index=False)
    pd.DataFrame(folds).to_csv(f"hmm_tema_yz_folds_{timestamp}.csv", index=False)
    validation_matrix.to_csv(f"hmm_tema_yz_validation_{timestamp}.csv")
    
    print(f"\nResults exported with timestamp: {timestamp}")
    print(f"  - Results: hmm_tema_yz_results_{timestamp}.csv")
    print(f"  - Trades: hmm_tema_yz_trades_{timestamp}.csv")
    print(f"  - Folds: hmm_tema_yz_folds_{timestamp}.csv")
    print(f"  - Validation: hmm_tema_yz_validation_{timestamp}.csv")
    
    print("\n" + "="*80)
    print("EXECUTION COMPLETE")
    print("="*80)
    
    return results, metrics, folds

if __name__ == "__main__":
    results, metrics, folds = main()